# 03 · Compare all completed models

Run after at least two model days are complete. Only compatible controlled
2-class runs are accepted. Missing or failed models remain visibly missing;
the notebook never creates placeholder metrics.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
REPO_PATH = Path("/content/aerial-object-detection-benchmark") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)], check=True)
    elif subprocess.check_output(["git", "-C", str(REPO_PATH), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Repository has local changes; refusing to update it.")
    else:
        subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "origin", "main"], check=True)
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/visdrone_architecture_benchmark")
    if IN_COLAB
    else Path(os.environ.get("VISDRONE_DRIVE_ROOT", REPO_PATH / "local_artifacts"))
)
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}


In [ ]:
if SMOKE_TEST:
    result = {"message": "Smoke mode: comparison requires at least two measured completed models."}
else:
    from src.workflows.comparison import compare_completed_models
    result = compare_completed_models(DRIVE_ROOT)
print(json.dumps(result, indent=2, default=str))


In [ ]:
if not SMOKE_TEST:
    from IPython.display import Image, Markdown, display
    report = Path(result["output"]) / "comparison.md"
    display(Markdown(report.read_text(encoding="utf-8")))
    figure = Path(result["output"]) / "accuracy_latency.png"
    if figure.is_file():
        display(Image(filename=str(figure)))
